# # Chicago Taxi Trips — Week of 11–17 September 2023
#
# *(Summary bullets will be inserted here once the analysis is complete.)*

# ## 1. Data loading and cleaning

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

FIG_DIR = "figures/analysis"
os.makedirs(FIG_DIR, exist_ok=True)

# Chart style: single blue, light grid, no top/right spines
BLUE = "#2a78d6"; ORANGE = "#eb6834"; GREY = "#9a9994"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
})

raw = pd.read_csv("trips.csv", dtype={"pickup_census_tract": "string", "dropoff_census_tract": "string"})
df = raw.copy()
print("Shape:", df.shape)
print("Duplicate trip_id:", df["trip_id"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Shape: (135768, 23)
Duplicate trip_id: 0
Fully duplicated rows: 0


In [2]:
print(df.dtypes)
print()
miss = df.isna().sum().to_frame("missing"); miss["pct"] = miss["missing"] / len(df) * 100
print(miss[miss.missing > 0])

trip_id                           str
taxi_id                           str
trip_start_timestamp              str
trip_end_timestamp                str
trip_seconds                  float64
trip_miles                    float64
pickup_census_tract            string
dropoff_census_tract           string
pickup_community_area         float64
dropoff_community_area        float64
fare                          float64
tips                          float64
tolls                         float64
extras                        float64
trip_total                    float64
payment_type                      str
company                           str
pickup_centroid_latitude      float64
pickup_centroid_longitude     float64
pickup_centroid_location          str
dropoff_centroid_latitude     float64
dropoff_centroid_longitude    float64
dropoff_centroid_location         str
dtype: object

                            missing   pct
trip_end_timestamp                1  0.00
trip_seconds               

In [3]:
num = ["trip_seconds","trip_miles","fare","tips","tolls","extras","trip_total"]
print(df[num].describe(percentiles=[.01,.05,.5,.95,.99]).T)

                  count     mean      std  min   1%    5%      50%      95%      99%       max
trip_seconds 135,735.00 1,346.54 1,762.25 0.00 0.00 22.00 1,020.00 3,382.30 4,872.98 86,160.00
trip_miles   135,768.00     6.94     7.85 0.00 0.00  0.00     3.24    18.76    27.14    792.90
fare         135,444.00    23.87    19.60 0.00 3.25  4.25    17.75    53.50    75.50    911.00
tips         135,444.00     3.38     4.67 0.00 0.00  0.00     2.00    11.88    17.10    130.00
tolls        135,444.00     0.13    20.42 0.00 0.00  0.00     0.00     0.00     0.00  4,444.44
extras       135,444.00     2.54    18.31 0.00 0.00  0.00     0.00     7.00    35.50  4,445.44
trip_total   135,444.00    30.12    43.93 0.00 3.25  5.25    21.25    70.50   104.43  8,901.13


In [ ]:
# Parse timestamps and normalise company names
df["start"] = pd.to_datetime(df["trip_start_timestamp"])
df["end"] = pd.to_datetime(df["trip_end_timestamp"])
print("Start range:", df["start"].min(), "->", df["start"].max())
print("Start minutes present:", sorted(df["start"].dt.minute.unique()))

# Variant spellings of the same company (identified by inspecting all 32 raw names)
company_map = {
    "Taxicab Insurance Agency, LLC": "Taxicab Insurance Agency Llc",
    "Choice Taxi Association Inc": "Choice Taxi Association",
}
df["company_raw"] = df["company"]
df["company"] = df["company"].replace(company_map)
print("Distinct company names: raw =", df["company_raw"].nunique(), "-> normalised =", df["company"].nunique())
print("Rows renamed:", (df["company"] != df["company_raw"]).sum())

In [ ]:
# Checks that are FLAGGED but not used to exclude rows (no business question uses miles or duration)
df["parts_sum"] = df[["fare","tips","tolls","extras"]].sum(axis=1, min_count=4)
df["total_gap"] = (df["trip_total"] - df["parts_sum"]).round(2)
mph = df["trip_miles"] / (df["trip_seconds"] / 3600)

checks = {
    "trip_seconds missing": df["trip_seconds"].isna(),
    "trip_seconds == 0": df["trip_seconds"] == 0,
    "trip_miles == 0": df["trip_miles"] == 0,
    "trip_miles > 100": df["trip_miles"] > 100,
    "trip_seconds > 12 h": df["trip_seconds"] > 12 * 3600,
    "implied speed > 100 mph": mph > 100,
    "trip_total - parts == 0.50 (card-fee-sized gap)": df["total_gap"] == 0.5,
    "trip_total - parts > 0.50 (unexplained gap)": df["total_gap"] > 0.5,
    "tips > fare": df["tips"] > df["fare"],
}
print(pd.Series({k: int(v.sum()) for k, v in checks.items()}, name="rows").to_frame().assign(pct=lambda t: t.rows / len(df) * 100))
print()
print("Any trip_total BELOW sum of parts?", int((df["total_gap"] < -0.01).sum()))
print("0.50 gap by payment type:\n", df.loc[df.total_gap == 0.5, "payment_type"].value_counts().to_string())
print("Unexplained gap sizes:\n", df.loc[df.total_gap > 0.5, "total_gap"].value_counts().to_string())

In [ ]:
# Are zero-mile trips real rides? Compare fare/duration of zero-mile trips with a duration to the rest
zm = (df["trip_miles"] == 0) & (df["trip_seconds"] > 60)
print("Zero-mile trips lasting > 1 min:", zm.sum())
print(df.loc[zm, ["fare","trip_seconds"]].describe().T[["count","50%","mean"]])
print("\nAll other trips with duration > 1 min:")
print(df.loc[~zm & (df["trip_seconds"] > 60), ["fare","trip_seconds"]].describe().T[["count","50%","mean"]])